In [ ]:
# Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx


# R5
import r5py
from r5py import TransportNetwork

# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon

import h3
from shapely.geometry import Polygon

In [ ]:
users = pd.read_parquet("./data/user_pois_bike_1_oulu.parquet")

In [ ]:
users.columns

In [ ]:
cols = [
    "Healthcare and Health",
    "Education",
    "Recreational, Outdoors",
    "Shopping, Errands",
    "Social, Cultural"
]

In [ ]:
# keep original values
users[[f"{c}_num" for c in cols]] = users[cols]

row_sum = users[cols].sum(axis=1)
users[cols] = users[cols].div(row_sum, axis=0) * 100

In [ ]:
# hsk_n = gpd.read_file("./data/districts/v2024.gpkg")

In [ ]:
finland_pc = gpd.read_file("./data/postal_code_finland/pno_tilasto_2024.shp")

In [ ]:
oulu_geo = gpd.read_file("./data/oulu_region_boundary.geojson")

In [ ]:
# 1. Make sure both are in the same CRS
finland_pc = finland_pc.to_crs(oulu_geo.crs)

# 2. Fix geometries if needed (important for intersections)
finland_pc["geometry"] = finland_pc.buffer(0)
oulu_geo["geometry"] = oulu_geo.buffer(0)

# 3. Compute original area of postal codes
finland_pc["area_total"] = finland_pc.geometry.area

# 4. Intersection with Turku
intersection = gpd.overlay(finland_pc, oulu_geo, how="intersection")

# 5. Compute intersection area
intersection["area_intersection"] = intersection.geometry.area

# 6. Aggregate intersection area per postal code
# ('postal_code_id' is the postal code identifier column)
area_ratio = (
    intersection.groupby("postinumer")["area_intersection"]
    .sum()
    .reset_index()
)

# 7. Merge back with original areas
area_ratio = area_ratio.merge(
    finland_pc[["postinumer", "area_total"]],
    on="postinumer"
)

# 8. Compute share of area inside Turku
area_ratio["share_inside"] = (
    area_ratio["area_intersection"] / area_ratio["area_total"]
)

# 9. Keep postal codes mostly inside Turku (e.g. >50%)
selected_ids = area_ratio.loc[
    area_ratio["share_inside"] > 0.5, "postinumer"
]

# 10. Filter original dataset
oulu_postcodes = finland_pc[
    finland_pc["postinumer"].isin(selected_ids)
]

In [ ]:
def h3_to_polygon(h3_id):
    return Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True))

users["geometry"] = users["home_gid9"].apply(h3_to_polygon)


In [ ]:
# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(users, geometry='geometry', crs="EPSG:4326")

In [ ]:
# Ensure both are in the same CRS
gdf = gdf.to_crs(oulu_postcodes.crs)

# Step 1: Compute intersections
intersections = gpd.overlay(gdf, oulu_postcodes, how='intersection')

# Step 2: Compute intersection area
intersections['intersect_area'] = intersections.geometry.area



In [ ]:
intersections.columns

In [ ]:
# Step 3: For each H3 hex, keep the postal code with the largest intersection
idx = intersections.groupby('home_gid9')['intersect_area'].idxmax()
largest_overlap = intersections.loc[idx]

# Step 4: Merge back the postal code to original H3 GeoDataFrame
gdf = gdf.merge(largest_overlap[['home_gid9', 'postinumer','nimi','geometry']], on='home_gid9', how='left')

In [ ]:
gdf = gdf.rename(columns={
    "nimi": "Nimi",
    "postinumer": "Posnro"
})

In [ ]:
gdf

In [ ]:
people_per_nimi = (
    gdf
    .groupby('Nimi')
    .size()
    .reset_index(name='n_people')
)

In [ ]:
people_per_nimi.describe()

In [ ]:
user_neighborhood = (
    gdf[['user_id', 'Nimi', 'Posnro']]
    .drop_duplicates(subset='user_id')
)

In [ ]:
gdf[gdf['is_home'] == 0]

In [ ]:
poi_cols = [
    'Healthcare and Health',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural',
]

nimi_poi_mean = (
    gdf
    .loc[gdf['is_home'] == 1]
    .groupby('Nimi')[poi_cols]
    .mean()
    .reset_index()
)

In [ ]:
visit_poi_mean = (
    gdf
    .loc[gdf['is_home'] == 0]
    .groupby('Nimi')[poi_cols]
    .mean()
    .reset_index()
)

In [ ]:
user_freq_counts = (
    gdf
    .loc[gdf['is_home'] != 1]   # exclude home rows
    # .loc[gdf['frequency_period'] > 2]  
    .groupby('user_id')
    .size()
    .reset_index(name='n_rows_freq_gt2')
)

In [ ]:
summary = {
    'n_users_rows_le_3': (user_freq_counts['n_rows_freq_gt2'] <= 3).sum(),
    'n_users_rows_gt_3': (user_freq_counts['n_rows_freq_gt2'] > 3).sum(),
    'n_users_rows_lt_5': (user_freq_counts['n_rows_freq_gt2'] < 5).sum(),
    'n_users_rows_ge_5': (user_freq_counts['n_rows_freq_gt2'] >= 5).sum(),
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['n_users'])

In [ ]:
user_freq_counts

In [ ]:
summary

In [ ]:
gdf_nh = gdf[gdf['is_home'] != 1].copy()

In [ ]:
gdf_nh_nw = gdf_nh[gdf_nh['is_work'] != 1].copy()

In [ ]:
poi_cols = [
    'Education',
    'Healthcare and Health',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]


In [ ]:
n_users = gdf_nh_nw['user_id'].nunique()
n_users

In [ ]:
users_2plus = (
    gdf_nh_nw
    .groupby('user_id')
    .size()
    .ge(2)          # ≥ 2 rows
    .sum()
)

users_2plus

In [ ]:
gdf_nh_nw = (
    gdf_nh_nw
    .groupby('user_id')
    .filter(lambda x: len(x) >= 2)
)

In [ ]:
gdf_nh_nw.columns


In [ ]:
gdf_nh_nw

In [ ]:
poi_num_cols = [
    'Education_num',
    'Healthcare and Health_num',
    'Recreational, Outdoors_num',
    'Shopping, Errands_num',
    'Social, Cultural_num'
]

user_activity_space = (
    gdf_nh_nw
    .groupby('user_id')[poi_num_cols]
    .sum()
    .reset_index()
)

In [ ]:
gdf_nh_nw = gdf_nh_nw.merge(
    user_activity_space,
    on='user_id',
    how='left',
    suffixes=('', '_activity')
)

In [ ]:
# Multiplying by frequency just in case

poi_num_cols = [
    'Education_num',
    'Healthcare and Health_num',
    'Recreational, Outdoors_num',
    'Shopping, Errands_num',
    'Social, Cultural_num'
]

user_activity_space_freq = (
    gdf_nh_nw
    .assign(**{
        c: gdf_nh_nw[c] * gdf_nh_nw['frequency_period']
        for c in poi_num_cols
    })
    .groupby('user_id')[poi_num_cols]
    .sum()
    .reset_index()
)

In [ ]:
user_activity_space_freq

In [ ]:
gdf_nh_nw = gdf_nh_nw.merge(
    user_activity_space_freq,
    on='user_id',
    how='left',
    suffixes=('', '_activity_freq')
)

“As the user visits POIs of this type (ordered from low to high CO₂ cost), how does cumulative CO₂ exposure build up?”

In [ ]:
gdf_nh_nw['co2_weighted'] = gdf_nh_nw['bike_co2_total'] * gdf_nh_nw['frequency_period']

In [ ]:
poi = 'Education'

# filter stays with at least 1 Education POI
df_edu = gdf_nh_nw.loc[gdf_nh_nw[f'{poi}_num'] > 0, ['user_id', 'co2_weighted', f'{poi}_num']].copy()

# explode rows by number of Education POIs in this stay
df_edu = df_edu.loc[df_edu.index.repeat(df_edu[f'{poi}_num'])]

# sort by CO2 ascending
df_edu = df_edu.sort_values(['user_id', 'co2_weighted'])

# cumulative visits per user
df_edu['cum_visits'] = df_edu.groupby('user_id').cumcount() + 1

# cumulative CO2 per user
df_edu['cum_co2'] = df_edu.groupby('user_id')['co2_weighted'].cumsum()

# normalize to 0-1 for plotting
df_edu['cum_visits_pct'] = df_edu['cum_visits'] / df_edu.groupby('user_id')['cum_visits'].transform('max')
df_edu['cum_co2_pct'] = df_edu['cum_co2'] / df_edu.groupby('user_id')['cum_co2'].transform('max')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

# select the user
user_id = 'fff81e73-8162-492e-9e7c-c9bed63db339'
user_df = df_edu[df_edu['user_id'] == user_id]

plt.figure(figsize=(8, 5))
plt.plot(
    user_df['cum_co2'],       # x-axis: cumulative CO2
    user_df['cum_visits_pct'],    # y-axis: cumulative share of visits
    marker='o', linestyle='-'
)

plt.xlabel("Cumulative share of CO₂")
plt.ylabel("Cumulative share of Education visits")
plt.title(f"Cumulative Education visits vs CO₂ for user {user_id}")
plt.grid(True)
plt.show()


In [ ]:
poi_cols = [
    'Education',
    'Healthcare and Health',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]

In [ ]:


sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

user_id = 'fff81e73-8162-492e-9e7c-c9bed63db339'

plt.figure(figsize=(10, 6))

for poi in poi_cols:
    # filter stays with at least 1 POI of this type
    df_poi = gdf_nh_nw.loc[gdf_nh_nw[f'{poi}_num'] > 0, ['user_id', 'co2_weighted', f'{poi}_num']].copy()
    df_poi = df_poi[df_poi['user_id'] == user_id]
    
    # explode by number of POIs
    df_poi = df_poi.loc[df_poi.index.repeat(df_poi[f'{poi}_num'])]
    
    # sort by CO2 ascending
    df_poi = df_poi.sort_values('co2_weighted')
    
    # cumulative visits & cumulative CO2
    df_poi['cum_visits'] = range(1, len(df_poi)+1)
    df_poi['cum_co2'] = df_poi['co2_weighted'].cumsum()
    
    # normalize 0-1 for plotting
    df_poi['cum_visits_pct'] = df_poi['cum_visits'] / df_poi['cum_visits'].max()
    
    # plot
    plt.plot(
        df_poi['cum_co2'],
        df_poi['cum_visits_pct'],
        marker='o', linestyle='-',
        label=poi
    )

plt.xlabel("Cumulative CO₂")
plt.ylabel("Cumulative share of visits")
plt.title(f"Cumulative visits vs CO₂ for user {user_id}")
plt.legend(title="POI Type")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# create co2 per trip (not weighted by frequency)
gdf_nh_nw['co2_per_trip'] = gdf_nh_nw['bike_co2_total']  # just the CO2 of the trip itself

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

user_id = 'fff81e73-8162-492e-9e7c-c9bed63db339'

plt.figure(figsize=(10, 6))

for poi in poi_cols:
    # filter stays with at least 1 POI of this type
    df_poi = gdf_nh_nw.loc[gdf_nh_nw[f'{poi}_num'] > 0, ['user_id', 'co2_per_trip', f'{poi}_num']].copy()
    df_poi = df_poi[df_poi['user_id'] == user_id]
    
    # explode by number of POIs
    df_poi = df_poi.loc[df_poi.index.repeat(df_poi[f'{poi}_num'])]
    
    # sort by CO2 ascending
    df_poi = df_poi.sort_values('co2_per_trip')
    
    # cumulative visits & cumulative CO2
    df_poi['cum_visits'] = range(1, len(df_poi)+1)
    df_poi['cum_co2'] = df_poi['co2_per_trip'].cumsum()
    
    # normalize visits for plotting
    df_poi['cum_visits_pct'] = df_poi['cum_visits'] / df_poi['cum_visits'].max()
    
    # plot
    plt.plot(
        df_poi['cum_co2'],
        df_poi['cum_visits_pct'],
        marker='o', linestyle='-',
        label=poi
    )

plt.xlabel("Cumulative CO₂ per trip")
plt.ylabel("Cumulative share of visits")
plt.title(f"Cumulative visits vs CO₂ per trip for user {user_id}")
plt.legend(title="POI Type")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

user_id = 'fff81e73-8162-492e-9e7c-c9bed63db339'

plt.figure(figsize=(10, 6))

for poi in poi_cols:
    # filter stays with at least 1 POI of this type
    df_poi = gdf_nh_nw.loc[gdf_nh_nw[f'{poi}_num'] > 0, ['user_id', 'co2_per_trip', f'{poi}_num']].copy()
    df_poi = df_poi[df_poi['user_id'] == user_id]
    
    # explode by number of POIs
    df_poi = df_poi.loc[df_poi.index.repeat(df_poi[f'{poi}_num'])]
    
    # sort by CO2 ascending
    df_poi = df_poi.sort_values('co2_per_trip')
    
    # cumulative visits & cumulative CO2
    df_poi['cum_visits'] = range(1, len(df_poi)+1)
    df_poi['cum_co2'] = df_poi['co2_per_trip'].cumsum()
    
    # normalize visits for plotting
    df_poi['cum_visits_pct'] = df_poi['cum_visits'] / df_poi['cum_visits'].max()
    
    # plot
    plt.plot(
        df_poi['cum_co2'],
        df_poi['cum_visits_pct'],
        marker='o', linestyle='-',
        label=poi
    )

plt.xlabel("Cumulative CO₂ per trip")
plt.ylabel("Cumulative share of visits")
plt.title(f"Cumulative visits vs CO₂ per trip for user {user_id}")
plt.legend(title="POI Type")
#plt.xlim(0, 2000)  # fix x-axis
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

user_id = 'fff81e73-8162-492e-9e7c-c9bed63db339'

plt.figure(figsize=(10, 6))

for poi in poi_cols:
    # filter stays with at least 1 POI of this type
    df_poi = gdf_nh_nw.loc[gdf_nh_nw[f'{poi}_num'] > 0, ['user_id', 'co2_per_trip', f'{poi}_num']].copy()
    df_poi = df_poi[df_poi['user_id'] == user_id]
    
    # explode by number of POIs
    df_poi = df_poi.loc[df_poi.index.repeat(df_poi[f'{poi}_num'])]
    
    # sort by CO2 ascending
    df_poi = df_poi.sort_values('co2_per_trip')
    
    # cumulative visits & cumulative CO2
    df_poi['cum_visits'] = range(1, len(df_poi)+1)
    df_poi['cum_co2'] = df_poi['co2_per_trip'].cumsum()
    
    # normalize visits for plotting
    df_poi['cum_visits_pct'] = df_poi['cum_visits'] / df_poi['cum_visits'].max()
    
    # plot
    plt.plot(
        df_poi['cum_co2'],
        df_poi['cum_visits_pct'],
        marker='o', linestyle='-',
        label=poi
    )

plt.xlabel("Cumulative CO₂ per trip")
plt.ylabel("Cumulative share of visits")
plt.title(f"Cumulative visits vs CO₂ per trip for user {user_id}")
plt.legend(title="POI Type")
plt.xlim(0, 2000)  # fix x-axis
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

user_id = 'fff81e73-8162-492e-9e7c-c9bed63db339'

plt.figure(figsize=(10, 6))

for poi in poi_cols:
    # filter stays with at least 1 POI of this type
    df_poi = gdf_nh_nw.loc[gdf_nh_nw[f'{poi}_num'] > 0, ['user_id', 'bike_co2_total', f'{poi}_num']].copy()
    df_poi = df_poi[df_poi['user_id'] == user_id]
    
    # explode by number of POIs
    df_poi = df_poi.loc[df_poi.index.repeat(df_poi[f'{poi}_num'])]
    
    # sort by CO2 ascending
    df_poi = df_poi.sort_values('bike_co2_total')
    
    # cumulative visits
    df_poi['cum_visits'] = range(1, len(df_poi)+1)
    df_poi['cum_visits_pct'] = df_poi['cum_visits'] / df_poi['cum_visits'].max()
    
    # running maximum CO2
    df_poi['cum_co2_max'] = df_poi['bike_co2_total'].cummax()
    
    # plot
    plt.plot(
        df_poi['cum_co2_max'],
        df_poi['cum_visits_pct'],
        marker='o', linestyle='-',
        label=poi
    )

plt.xlabel("Running max CO₂ per trip")
plt.ylabel("Cumulative share of visits")
plt.title(f"Cumulative visits vs running max CO₂ per trip for user {user_id}")
plt.legend(title="POI Type")
plt.xlim(0, 1500)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

records = []

for poi in poi_cols:
    df_poi = gdf_nh_nw.loc[
        gdf_nh_nw[f'{poi}_num'] > 0,
        ['user_id', 'bike_co2_total', f'{poi}_num']
    ].copy()
    
    # explode by number of POIs
    df_poi = df_poi.loc[df_poi.index.repeat(df_poi[f'{poi}_num'])]

    # Keep only users with at least 3 points for this POI
    df_poi = (
        df_poi
        .groupby('user_id')
        .filter(lambda x: len(x) >= 5)
    )

    # sort by CO2 ascending within each user
    df_poi = df_poi.sort_values(['user_id', 'bike_co2_total'])

    # cumulative visits
    df_poi['cum_visits'] = df_poi.groupby('user_id').cumcount() + 1
    df_poi['cum_visits_pct'] = (
        df_poi['cum_visits'] /
        df_poi.groupby('user_id')['cum_visits'].transform('max')
    )

    # running max CO2
    df_poi['cum_co2_max'] = (
        df_poi.groupby('user_id')['bike_co2_total'].cummax()
    )

    df_poi['poi_type'] = poi

    records.append(
        df_poi[['user_id', 'poi_type', 'cum_visits_pct', 'cum_co2_max']]
    )

df_cummax_all = pd.concat(records, ignore_index=True)



In [ ]:
df_cummax_all

In [ ]:
df_cummax_all.groupby(['poi_type', 'user_id']).size().describe()


### Doing from here BY USER and not by NH

In [ ]:
df_cummax_all

In [ ]:
# get unique user -> neighborhood mapping
user_neighborhood = (
    gdf_nh_nw[['user_id', 'Nimi', 'Posnro']]
    .drop_duplicates(subset=['user_id'])
)

# merge with df_cummax_all
df_cummax_all = df_cummax_all.merge(user_neighborhood, on='user_id', how='left')

# check
df_cummax_all.head()


In [ ]:
# Count distinct cum_co2_max per (user_id, poi_type)
group_counts = (
    df_cummax_all
    .groupby(["user_id", "poi_type"])["cum_co2_max"]
    .nunique()
    .reset_index(name="n_unique")
)

# Keep only groups with >= 3 unique cum_co2_max
valid_groups = group_counts[group_counts["n_unique"] >= 3]

# Filter original dataframe
df_cummax_filtered = df_cummax_all.merge(
    valid_groups[["user_id", "poi_type"]],
    on=["user_id", "poi_type"],
    how="inner"
)

In [ ]:
group_counts

In [ ]:
df_cummax_filtered

### JOBS

In [ ]:
import pyarrow.parquet as pq

cols_needed = [
    "from_id","to_id", "co2_emissions_g"
]

table = pq.read_table("scratch/bike_co2_3000_oulu.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_bike_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
df_bike_co2

In [ ]:
df_bike_co2_sym = df_bike_co2.merge(
    df_bike_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)

In [ ]:
# Outbound trip
df_bike_co2_sym["bike_co2_outbound"] = df_bike_co2_sym["co2_emissions_g_outbound"]

# Inbound trip (fill missing with outbound if return trip not in table)
df_bike_co2_sym["bike_co2_inbound"] = (
    df_bike_co2_sym["co2_emissions_g_inbound"]
    .fillna(df_bike_co2_sym["co2_emissions_g_outbound"])
)

# Total round-trip emissions
df_bike_co2_sym["bike_co2_total"] = (
    df_bike_co2_sym["bike_co2_outbound"] + df_bike_co2_sym["bike_co2_inbound"]
)


In [ ]:
df_bike_co2_final = (
    df_bike_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "bike_co2_outbound",
            "bike_co2_inbound",
            "bike_co2_total"
        ]
    ]
)


In [ ]:
df_bike_co2 = df_bike_co2_final.copy()

In [ ]:
df_jobs = pd.read_parquet("./data/users_and_works_oulu.parquet")

In [ ]:
df_jobs

In [ ]:
df_jobs_co2 = df_jobs.merge(
    df_bike_co2,
    left_on=["home_gid9", "work_gid9"],
    right_on=["from_id", "to_id"],
    how="left"
)


In [ ]:
df_jobs_co2

In [ ]:
df_typical = (
    df_cummax_all
    .loc[df_cummax_all["cum_visits_pct"] >= 0.5]
    .sort_values(["user_id", "poi_type", "cum_visits_pct"])
    .groupby(
        ["user_id", "poi_type", "Nimi", "Posnro"],
        as_index=False
    )
    .first()
    .rename(columns={"cum_co2_max": "typical_trip_co2"})
)

In [ ]:
# Keep user info
user_info = df_typical[['user_id', 'Nimi', 'Posnro']].drop_duplicates(subset='user_id')

# Create bike "jobs" rows
df_jobs_typical_bike = (
    df_jobs_co2
    .merge(user_info, on="user_id", how="left")  # get Nimi and Posnro
    .assign(
        poi_type="jobs",
        cum_visits_pct=1,
        typical_trip_co2=lambda x: x["bike_co2_total"]
    )[
        ['user_id', 'poi_type', 'Nimi', 'Posnro', 'cum_visits_pct', 'typical_trip_co2']
    ]
)

# Combine with existing typical data
df_typical_with_jobs_bike = pd.concat([df_typical, df_jobs_typical_bike], ignore_index=True)



In [ ]:
df_typical = df_typical = df_typical_with_jobs_bike.copy()

In [ ]:
# -----------------------------------------------------------
# SENSITIVITY COLUMNS: typical_trip_co2 at thresholds 0–100
# _0  = minimum cum_co2_max per (user_id, poi_type) (not literal 0)
# _50 = same logic as existing typical_trip_co2 (0.5 threshold)
# jobs rows: always car_co2_total regardless of threshold
# -----------------------------------------------------------

thresholds = [i / 100 for i in range(0, 101, 10)]  # 0.0, 0.1, ..., 1.0

# Minimum per (user_id, poi_type) — used for threshold 0
min_co2 = (
    df_cummax_all
    .groupby(["user_id", "poi_type", "Nimi", "Posnro"], as_index=False)["cum_co2_max"]
    .min()
    .rename(columns={"cum_co2_max": "typical_trip_co2_0"})
)

# Build one column per threshold
sensitivity_cols = {}

for t in thresholds:
    col = f"typical_trip_co2_{int(round(t * 100))}"

    if t == 0:
        # Already computed as minimum
        sensitivity_cols[col] = min_co2.set_index(["user_id", "poi_type"])["typical_trip_co2_0"]
        continue

    df_t = (
        df_cummax_all
        .loc[df_cummax_all["cum_visits_pct"] >= t]
        .sort_values(["user_id", "poi_type", "cum_visits_pct"])
        .groupby(["user_id", "poi_type", "Nimi", "Posnro"], as_index=False)
        .first()[["user_id", "poi_type", "cum_co2_max"]]
        .rename(columns={"cum_co2_max": col})
        .set_index(["user_id", "poi_type"])[col]
    )
    sensitivity_cols[col] = df_t

# Assemble into a single dataframe indexed by (user_id, poi_type)
df_sensitivity = pd.DataFrame(sensitivity_cols).reset_index()
df_sensitivity.columns.name = None

# Add jobs rows: all threshold columns = car_co2_total
df_jobs_sensitivity = (
    df_jobs_co2
    .merge(user_info, on="user_id", how="left")
    .assign(poi_type="jobs")
    [["user_id", "poi_type"]]
)
for t in thresholds:
    col = f"typical_trip_co2_{int(round(t * 100))}"
    df_jobs_sensitivity[col] = df_jobs_co2["bike_co2_total"].values

df_sensitivity = pd.concat([df_sensitivity, df_jobs_sensitivity], ignore_index=True)

print("Sensitivity columns added:", [c for c in df_sensitivity.columns if c.startswith("typical_trip_co2_")])
print("Shape:", df_sensitivity.shape)
df_sensitivity.head()


In [ ]:
# Merge sensitivity columns into df_typical and save
df_typical_enriched = df_typical.merge(
    df_sensitivity,
    on=["user_id", "poi_type"],
    how="left"
)

df_typical_enriched.to_parquet("scratch/bike_typ_cat_oulu.parquet")
print("Saved. Columns:", list(df_typical_enriched.columns))
print("Shape:", df_typical_enriched.shape)

In [ ]:
df_typical_enriched

In [ ]:
#df_typical.to_parquet("scratch/bike_typ_cat_oulu.parquet")

In [ ]:
df_typical.sort_values("typical_trip_co2").head(250000)

In [ ]:
# Define thresholds (grams CO2 per trip)
thresholds = {
    "3km": 64*2,
    "5km": 107*2,
    "10km": 214*2
}

# Function to calculate % below threshold per poi_type, ignoring NaNs
percent_below = pd.DataFrame({
    th: df_typical.groupby("poi_type")["typical_trip_co2"].apply(
        lambda x: (x.dropna() <= val).mean() * 100
    )
    for th, val in thresholds.items()
})

percent_below = percent_below.reset_index()
percent_below



In [ ]:
worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

student_profile = {
    "Education": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
def filter_complete_users(df_typical, profile):
    required_pois = set(profile.keys())

    valid_users = (
        df_typical
        .groupby("user_id")["poi_type"]
        .apply(set)
        .loc[lambda s: s.apply(lambda x: required_pois.issubset(x))]
        .index
    )

    return df_typical[df_typical["user_id"].isin(valid_users)]

In [ ]:
def profile_to_df(profile):
    return (
        pd.DataFrame(profile.items(), columns=["poi_type", "weekly_visits"])
    )

In [ ]:
def compute_weekly_emissions(df_typical, profile):

    profile_df = pd.DataFrame(
        profile.items(),
        columns=["poi_type", "weekly_visits"]
    )

    df_weekly = (
        df_typical
        .merge(profile_df, on="poi_type", how="inner")
        .assign(
            weekly_co2=lambda x: x["typical_trip_co2"] * x["weekly_visits"]
        )
    )

    return (
        df_weekly
        .groupby(
            ["user_id", "Nimi", "Posnro"],
            as_index=False
        )["weekly_co2"]
        .sum()
        .rename(columns={"weekly_co2": "total_weekly_co2"})
    )


In [ ]:
df_typical_workers = filter_complete_users(
    df_typical,
    worker_profile
)

weekly_worker_co2 = compute_weekly_emissions(
    df_typical_workers,
    worker_profile
)

In [ ]:
neighborhood_co2 = (
    weekly_worker_co2
    .groupby(["Nimi", "Posnro"], as_index=False)
    .agg(
        mean_weekly_co2=("total_weekly_co2", "mean"),
        median_weekly_co2=("total_weekly_co2", "median"),
        q25_weekly_co2=("total_weekly_co2", lambda x: x.quantile(0.25)),
        q75_weekly_co2=("total_weekly_co2", lambda x: x.quantile(0.75)),
        n_users=("user_id", "nunique")
    )
)

In [ ]:
neighborhood_co2.sort_values("mean_weekly_co2").head(50)

In [ ]:
weekly_worker_co2.to_parquet("./output/bike_expenditure_weekly_typical_oulu.parquet")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Data ---
df = weekly_worker_co2.copy()
df = df[df["total_weekly_co2"].notna()]

# Optional: remove extreme outliers for readability
df = df[df["total_weekly_co2"] <= df["total_weekly_co2"].quantile(0.99)]

values = df["total_weekly_co2"]

# --- Figure ---
fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    values,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# --- Climate budgets ---
ax.axvline(
    7000,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    3000,
    color="tab:red",
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# --- Formatting ---
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ emissions per worker (grams)", fontsize=11)
ax.set_title(
    "Distribution of Weekly Worker CO₂ Emissions\nCompared with Climate Budgets",
    fontsize=14
)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()



In [ ]:

# Budgets (grams CO2 / week)
BUDGET_2030 = 7000
BUDGET_2050 = 3000

total_workers = len(df)

pct_below_2030 = (df["total_weekly_co2"] <= BUDGET_2030).mean() * 100
pct_below_2050 = (df["total_weekly_co2"] <= BUDGET_2050).mean() * 100

pct_below_2030, pct_below_2050

fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    df["total_weekly_co2"],
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# Budget lines
ax.axvline(
    BUDGET_2030,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    BUDGET_2050,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# Annotations below the boxplot (using axes coordinates)
ax.text(
    0.75,   # x = 75% of axes width
    -0.1,   # y = just below the boxplot
    f"{pct_below_2030:.1f}% below 2030 budget",
    color="tab:orange",
    fontsize=10,
    ha="left",
    va="top",
    transform=ax.transAxes  # now coordinates are relative to axes
)

ax.text(
    0.75,
    -0.15,  # slightly lower than the first annotation
    f"{pct_below_2050:.1f}% below 2050 budget",
    color="tab:blue",
    fontsize=10,
    ha="left",
    va="top",
    transform=ax.transAxes
)

# Formatting
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ emissions per worker (grams)", fontsize=11)
ax.set_title(
    "Weekly Worker CO₂ Emissions Compared with Climate Budgets",
    fontsize=14
)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Convert to kg
values_kg = df["total_weekly_co2"].dropna() / 1000  # grams → kg

# Optional: remove extreme outliers for readability
values_kg = values_kg[values_kg <= values_kg.quantile(0.99)]

# Histogram / frequency distribution
fig, ax = plt.subplots(figsize=(10, 4))

bins = np.arange(0, values_kg.max() + 0.5, 0.5)  # bin size = 0.5 kg
ax.hist(values_kg, bins=bins, color="lightgray", edgecolor="black")

# Budget lines (kg)
BUDGET_2030_KG = 7
BUDGET_2050_KG = 3
ax.axvline(BUDGET_2030_KG, color="tab:orange", linestyle="--", linewidth=2,
           label="2030 budget (7 kg/week)")
ax.axvline(BUDGET_2050_KG, color="tab:blue", linestyle="--", linewidth=2,
           label="2050 budget (3 kg/week)")

# Percentages below budgets (axes coords for lower-right)
pct_below_2030 = (values_kg <= BUDGET_2030_KG).mean() * 100
pct_below_2050 = (values_kg <= BUDGET_2050_KG).mean() * 100

ax.text(0.75, 0.85, f"{pct_below_2030:.1f}% ≤ 2030 budget", color="tab:orange",
        fontsize=10, ha="left", va="top", transform=ax.transAxes)
ax.text(0.75, 0.78, f"{pct_below_2050:.1f}% ≤ 2050 budget", color="tab:blue",
        fontsize=10, ha="left", va="top", transform=ax.transAxes)

# Formatting
ax.set_xlabel("Weekly CO₂ emissions per worker (kg)", fontsize=11)
ax.set_ylabel("Number of workers")
ax.set_title("Distribution of Weekly Worker CO₂ Emissions", fontsize=14)
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
## Weight by population

In [ ]:
census = pd.read_csv(
    "./data/pop_by_postal.csv",
    encoding="latin1"
)

In [ ]:
census["Posnro"] = census["Postal code area"].str.slice(0, 5)
census["Nimi"] = census["Postal code area"].str.slice(6).str.strip()

In [ ]:
census

In [ ]:
neighborhood_co2 = neighborhood_co2.merge(
    census[["Posnro", "2024"]],
    on="Posnro",
    how="left"
)

In [ ]:
neighborhood_co2 = neighborhood_co2.rename(
    columns={"2024": "population_2024"}
)

In [ ]:
neighborhood_co2.sort_values("population_2024")

In [ ]:
# Intersections
overlay = gpd.overlay(area_proj, census, how="intersection")

# Calculate intersection area
overlay["inter_area"] = overlay.geometry.area

# Weight population by share of area
overlay["weighted_pop"] = overlay["he_vakiy"] * (overlay["inter_area"] / overlay.groupby("id_nro")["inter_area"].transform("sum"))

# Aggregate back to hexagons
hex_with_census = overlay.groupby("ID").agg(
    {"weighted_pop": "sum"}  # keep hex geometry
)

In [ ]:
# Convert H3 ids to hexagon geometries
hex_with_census["geometry"] = hex_with_census.index.to_series().apply(
    lambda h: Polygon(h3.h3_to_geo_boundary(str(h), geo_json=True))
)

# Convertir en GeoDataFrame
area_with_id = gpd.GeoDataFrame(hex_with_census, geometry="geometry", crs="EPSG:4326")

In [ ]:
area_with_id

In [ ]:
# get unique user -> neighborhood mapping
user_neighborhood = (
    gdf_nh_nw[['user_id', 'Nimi', 'Posnro']]
    .drop_duplicates(subset=['user_id'])
)

# merge with df_cummax_all
df_cummax_all = df_cummax_all.merge(user_neighborhood, on='user_id', how='left')

# check
df_cummax_all.head()


In [ ]:
neighborhoods = ['Tapanila', 'Punavuori', 'Kruununhaka', 'Etelä-Haaga']

import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

plt.figure(figsize=(12, 6))

for nimi in neighborhoods:  # only these neighborhoods
    for poi in poi_cols:
        df_poi = df_cummax_all[(df_cummax_all['Nimi'] == nimi) &
                               (df_cummax_all['poi_type'] == poi)]
        
        if df_poi.empty:
            continue  # skip if no data
        
        # only keep relevant columns
        x = df_poi['cum_co2_max'].values
        y = df_poi['cum_visits_pct'].values
        
        # LOWESS smoothing
        lowess_smoothed = sm.nonparametric.lowess(y, x, frac=0.1)  # adjust frac for smoothness
        
        plt.plot(lowess_smoothed[:, 0], lowess_smoothed[:, 1], label=f"{poi} ({nimi})")
    
plt.xlabel("Running max CO₂ per trip")
plt.ylabel("Cumulative share of visits")
plt.title("Smoothed cumulative visits vs CO₂ per POI (Selected Neighborhoods)")
plt.xlim(0,750)
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)

# neighborhoods to plot
neighborhoods = ['Tapanila', 'Punavuori', 'Kruununhaka', 'Etelä-Haaga']

for nimi in neighborhoods:
    plt.figure(figsize=(10, 6))
    
    for poi in poi_cols:
        df_poi = df_cummax_all[(df_cummax_all['Nimi'] == nimi) &
                               (df_cummax_all['poi_type'] == poi)]
        
        if df_poi.empty:
            continue  # skip if no data
        
        x = df_poi['cum_co2_max'].values
        y = df_poi['cum_visits_pct'].values
        
        # LOWESS smoothing
        lowess_smoothed = sm.nonparametric.lowess(y, x, frac=0.1)
        
        plt.plot(lowess_smoothed[:, 0], lowess_smoothed[:, 1], label=f"{poi}")
    
    plt.xlabel("Running max CO₂ per trip")
    plt.ylabel("Cumulative share of visits")
    plt.title(f"Smoothed cumulative visits vs CO₂ in {nimi}")
    plt.xlim(0, 500)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend(title="POI Type")
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import pandas as pd

# neighborhoods and POIs of interest

neighborhoods = ['Tapanila', 'Punavuori', 'Kruununhaka', 'Etelä-Haaga', "Niittykumpu"]
poi_cols = [
    'Education',
    'Healthcare and Health',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]

results = []

for nimi in neighborhoods:
    for poi in poi_cols:
        
        # subset data
        df_poi = df_cummax_all[
            (df_cummax_all['Nimi'] == nimi) &
            (df_cummax_all['poi_type'] == poi)
        ].copy()
        
        if df_poi.empty:
            continue
        
        # keep only users with >= 3 points
        valid_users = (
            df_poi.groupby('user_id')
            .size()
            .loc[lambda x: x >= 4]
            .index
        )
        df_poi = df_poi[df_poi['user_id'].isin(valid_users)]
        
        if df_poi.empty:
            continue
        
        # define CO2 grid up to pooled 95th percentile
        co2_max = np.percentile(df_poi['cum_co2_max'], 95)
        co2_grid = np.linspace(0, co2_max, 100)
        
        # interpolate individual user curves
        curves = []
        for user_id, g in df_poi.groupby('user_id'):
            g = g.sort_values('cum_co2_max')
            curves.append(
                np.interp(
                    co2_grid,
                    g['cum_co2_max'],
                    g['cum_visits_pct'],
                    left=0,
                    right=1
                )
            )
        
        curves = np.vstack(curves)
        
        # aggregate
        median_curve = np.median(curves, axis=0)
        p25 = np.percentile(curves, 25, axis=0)
        p75 = np.percentile(curves, 75, axis=0)
        
        # --- OPTION 1: 90% saturation cutoff ---
        idx_90 = np.where(median_curve >= 0.9)[0]
        if len(idx_90) == 0:
            continue
        
        cutoff_idx = idx_90[0]
        co2_90 = co2_grid[cutoff_idx]
        
        # store truncated curve
        results.append(
            pd.DataFrame({
                'Nimi': nimi,
                'poi_type': poi,
                'cum_co2': co2_grid[:cutoff_idx + 1],
                'median_visits': median_curve[:cutoff_idx + 1],
                'p25_visits': p25[:cutoff_idx + 1],
                'p75_visits': p75[:cutoff_idx + 1],
                'co2_90': co2_90
            })
        )

# final dataset
df_neighborhood_curves = pd.concat(results, ignore_index=True)

df_neighborhood_curves.head()



In [ ]:
df_neighborhood_curves

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.1)

neighborhoods = df_neighborhood_curves['Nimi'].unique()
poi_order = [
    'Education',
    'Healthcare and Health',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]

for nimi in neighborhoods:
    
    df_n = df_neighborhood_curves[df_neighborhood_curves['Nimi'] == nimi]
    
    plt.figure(figsize=(10, 6))
    
    for poi in poi_order:
        df_p = df_n[df_n['poi_type'] == poi]
        if df_p.empty:
            continue
        
        # median curve
        plt.plot(
            df_p['cum_co2'],
            df_p['median_visits'],
            label=poi,
            linewidth=2
        )
        
        # uncertainty band (IQR)
        plt.fill_between(
            df_p['cum_co2'],
            df_p['p25_visits'],
            df_p['p75_visits'],
            alpha=0.2
        )
    
    # dynamic x-limit (max CO2 needed to reach ~90%)
    x_max = df_n['cum_co2'].max()
    
    plt.xlim(0, x_max)
    plt.ylim(0, 1)
    
    plt.xlabel("Running max CO₂ per trip")
    plt.ylabel("Cumulative share of visits")
    plt.title(f"Cumulative opportunity–CO₂ curves\n{nimi}")
    
    plt.legend(title="POI type")
    plt.tight_layout()
    plt.show()


In [ ]:
df_n = df_neighborhood_curves[df_neighborhood_curves['Nimi'] == nimi]

In [ ]:
df_n

In [ ]:
import numpy as np
import pandas as pd

# Get all unique neighbourhoods
neighborhoods = df_cummax_all['Nimi'].dropna().unique()
poi_cols = [
    'Education',
    'Healthcare and Health',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]

results = []

for nimi in neighborhoods:
    for poi in poi_cols:
        
        # subset data for this neighborhood and POI
        df_poi = df_cummax_all[
            (df_cummax_all['Nimi'] == nimi) &
            (df_cummax_all['poi_type'] == poi)
        ].copy()
        
        if df_poi.empty:
            continue
        
        # keep only users with >= 3 points for this POI
        valid_users = (
            df_poi.groupby('user_id')
            .size()
            .loc[lambda x: x >= 3]
            .index
        )
        df_poi = df_poi[df_poi['user_id'].isin(valid_users)]
        if df_poi.empty:
            continue
        
        # define CO2 grid up to 95th percentile for this neighborhood and POI
        co2_max = np.percentile(df_poi['cum_co2_max'], 95)
        co2_grid = np.linspace(0, co2_max, 100)
        
        # interpolate individual user curves onto common grid
        curves = []
        for user_id, g in df_poi.groupby('user_id'):
            g = g.sort_values('cum_co2_max')
            curves.append(
                np.interp(
                    co2_grid,
                    g['cum_co2_max'],
                    g['cum_visits_pct'],
                    left=0,
                    right=1
                )
            )
        curves = np.vstack(curves)
        
        # aggregate curves
        median_curve = np.median(curves, axis=0)
        p25 = np.percentile(curves, 25, axis=0)
        p75 = np.percentile(curves, 75, axis=0)
        
        # truncate at 90% of cumulative visits
        idx_90 = np.where(median_curve >= 0.9)[0]
        if len(idx_90) == 0:
            continue
        cutoff_idx = idx_90[0]
        co2_90 = co2_grid[cutoff_idx]
        
        # store results
        results.append(
            pd.DataFrame({
                'Nimi': nimi,
                'poi_type': poi,
                'cum_co2': co2_grid[:cutoff_idx + 1],
                'median_visits': median_curve[:cutoff_idx + 1],
                'p25_visits': p25[:cutoff_idx + 1],
                'p75_visits': p75[:cutoff_idx + 1],
                'co2_90': co2_90
            })
        )

# combine all results into one DataFrame
df_neighborhood_curves_all = pd.concat(results, ignore_index=True)

# check the final result
df_neighborhood_curves_all.head()


In [ ]:
# define threshold for "median reached"
threshold = 0.5

df_median_co2 = (
    df_neighborhood_curves_all
    .loc[lambda d: d['median_visits'] >= threshold]
    .groupby(['Nimi', 'poi_type'], as_index=False)
    .first()[['Nimi', 'poi_type', 'cum_co2']]
    .rename(columns={'cum_co2': 'co2_at_median'})
)

df_neighborhood_poi_median = (
    df_median_co2
    .pivot(index='Nimi', columns='poi_type', values='co2_at_median')
    .reset_index()
)

In [ ]:
df_neighborhood_poi_median.sort_values("Shopping, Errands").head(50)

### ISOTONIC

In [ ]:
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression

results = []

for nimi in df_cummax_all['Nimi'].unique():
    for poi in df_cummax_all['poi_type'].unique():
        
        # subset data
        df_poi = df_cummax_all[
            (df_cummax_all['Nimi'] == nimi) &
            (df_cummax_all['poi_type'] == poi)
        ].copy()
        
        if df_poi.empty:
            continue
        
        # keep only users with >= 4 points
        valid_users = (
            df_poi.groupby('user_id')
            .size()
            .loc[lambda x: x >= 4]
            .index
        )
        df_poi = df_poi[df_poi['user_id'].isin(valid_users)]
        
        if df_poi.empty:
            continue
        
        # define CO2 grid up to pooled 95th percentile
        co2_max = np.percentile(df_poi['cum_co2_max'], 95)
        co2_grid = np.linspace(0, co2_max, 100)
        
        curves = []
        
        for user_id, g in df_poi.groupby('user_id'):
            g = g.sort_values('cum_co2_max')
            
            # -------------------------------
            # ANCHOR AT (0, 0)
            # -------------------------------
            x = np.concatenate([[0], g['cum_co2_max'].values])
            y = np.concatenate([[0], g['cum_visits_pct'].values])
            
            iso = IsotonicRegression(
                increasing=True,
                y_min=0,
                y_max=1,
                out_of_bounds='clip'
            )
            
            y_iso = iso.fit_transform(x, y)
            
            # evaluate isotonic curve on common CO2 grid
            y_grid = np.interp(
                co2_grid,
                x,
                y_iso,
                left=0,
                right=1
            )
            
            curves.append(y_grid)
        
        curves = np.vstack(curves)
        
        # aggregate across users
        median_curve = np.median(curves, axis=0)
        p25 = np.percentile(curves, 25, axis=0)
        p75 = np.percentile(curves, 75, axis=0)
        
        # --- 90% saturation cutoff ---
        idx_90 = np.where(median_curve >= 0.9)[0]
        if len(idx_90) == 0:
            continue
        
        cutoff_idx = idx_90[0]
        co2_90 = co2_grid[cutoff_idx]
        
        # store results
        results.append(
            pd.DataFrame({
                'Nimi': nimi,
                'poi_type': poi,
                'cum_co2': co2_grid[:cutoff_idx + 1],
                'median_visits': median_curve[:cutoff_idx + 1],
                'p25_visits': p25[:cutoff_idx + 1],
                'p75_visits': p75[:cutoff_idx + 1],
                'co2_90': co2_90
            })
        )

# final dataset
df_neighborhood_curves_iso = pd.concat(results, ignore_index=True)

df_neighborhood_curves_iso.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.1)

neighborhoods = [
    'Tapanila',
    'Punavuori',
    'Kruununhaka',
    'Etelä-Haaga',
    'Niittykumpu'
]

for nimi in neighborhoods:
    
    df_nh = df_neighborhood_curves_iso[
        df_neighborhood_curves_iso['Nimi'] == nimi
    ]
    
    if df_nh.empty:
        continue
    
    plt.figure(figsize=(10, 6))
    
    for poi, g in df_nh.groupby('poi_type'):
        g = g.sort_values('cum_co2')
        
        plt.plot(
            g['cum_co2'],
            g['median_visits'],
            label=poi,
            linewidth=2
        )
        
        # uncertainty band
        plt.fill_between(
            g['cum_co2'],
            g['p25_visits'],
            g['p75_visits'],
            alpha=0.2
        )
    
    plt.xlabel("Running maximum CO₂ per trip")
    plt.ylabel("Cumulative share of visits")
    plt.title(f"Cumulative accessibility curves — {nimi}")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend(title="POI type")
    plt.tight_layout()
    plt.show()

# combine all results into one DataFrame
df_neighborhood_curves_all = pd.concat(results, ignore_index=True)

# check the final result
df_neighborhood_curves_all.head()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression

results = []

for nimi in df_cummax_all['Nimi'].unique():
    for poi in df_cummax_all['poi_type'].unique():

        # subset neighborhood + POI
        df_poi = df_cummax_all[
            (df_cummax_all['Nimi'] == nimi) &
            (df_cummax_all['poi_type'] == poi)
        ].copy()

        if df_poi.empty:
            continue

        # keep users with >= 3 exposed POIs
        valid_users = (
            df_poi.groupby('user_id')
            .size()
            .loc[lambda x: x >= 3]
            .index
        )
        df_poi = df_poi[df_poi['user_id'].isin(valid_users)]

        if df_poi.empty:
            continue

        # CO2 grid (robust upper bound)
        co2_max = np.percentile(df_poi['cum_co2_max'], 95)
        co2_grid = np.linspace(0, co2_max, 100)

        curves = []

        for user_id, g in df_poi.groupby('user_id'):
            g = g.sort_values('cum_co2_max')

            x = g['cum_co2_max'].values
            y = g['cum_visits_pct'].values

            # force origin
            x = np.insert(x, 0, 0.0)
            y = np.insert(y, 0, 0.0)

            iso = IsotonicRegression(
                increasing=True,
                y_min=0,
                y_max=1,
                out_of_bounds='clip'
            )

            y_iso = iso.fit_transform(x, y)

            # evaluate on common grid
            y_grid = np.interp(
                co2_grid,
                x,
                y_iso,
                left=0,
                rightconsidering=1
            )

            curves.append(y_grid)

        curves = np.vstack(curves)

        median_curve = np.median(curves, axis=0)
        p25 = np.percentile(curves, 25, axis=0)
        p75 = np.percentile(curves, 75, axis=0)

        # 90% saturation point
        idx_90 = np.where(median_curve >= 0.9)[0]
        if len(idx_90) == 0:
            continue

        cutoff_idx = idx_90[0]
        co2_90 = co2_grid[cutoff_idx]

        results.append(
            pd.DataFrame({
                'Nimi': nimi,
                'poi_type': poi,
                'cum_co2': co2_grid[:cutoff_idx + 1],
                'median_visits': median_curve[:cutoff_idx + 1],
                'p25_visits': p25[:cutoff_idx + 1],
                'p75_visits': p75[:cutoff_idx + 1],
                'co2_90': co2_90
            })
        )

# combine all results into one DataFrame
df_neighborhood_curves_all = pd.concat(results, ignore_index=True)

# check the final result
df_neighborhood_curves_all.head()


In [ ]:
import numpy as np

df_decay = []

for (nimi, poi), g in df_neighborhood_curves.groupby(['Nimi', 'poi_type']):
    
    g = g.sort_values('cum_co2').copy()
    
    # numerical derivative
    g['decay'] = g['median_visits'].diff() / g['cum_co2'].diff()
    
    # clean
    g['decay'] = g['decay'].clip(lower=0)
    g = g.dropna(subset=['decay'])
    
    # normalize to integrate to 1
    area = np.trapz(g['decay'], g['cum_co2'])
    if area > 0:
        g['decay'] = g['decay'] / area
    
    df_decay.append(
        g[['Nimi', 'poi_type', 'cum_co2', 'decay']]
    )

df_decay = pd.concat(df_decay, ignore_index=True)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.1)

for nimi in df_decay['Nimi'].unique():
    
    plt.figure(figsize=(10, 5))
    
    df_n = df_decay[df_decay['Nimi'] == nimi]
    
    for poi in poi_cols:
        df_p = df_n[df_n['poi_type'] == poi]
        if df_p.empty:
            continue
        
        plt.plot(
            df_p['cum_co2'],
            df_p['decay'],
            label=poi,
            linewidth=2
        )
    
    plt.xlabel("CO₂ cost per trip")
    plt.ylabel("Accessibility decay")
    plt.title(f"CO₂-based accessibility decay\n{nimi}")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

gdf_nh['pt_co2_total_weighted'] = (
    gdf_nh['pt_co2_total'] * gdf_nh['frequency_period'])


# Step 1: Fractionally allocate CO₂ to each POI type

for c in poi_cols:
    gdf_nh[f'co2_{c}'] = (
        gdf_nh['pt_co2_total_weighted'] * (gdf_nh[c] / 100)
    )


In [ ]:
poi_num_cols = [f'{c}_num' for c in poi_cols]

for c in poi_num_cols:
    gdf_nh[f'exp_{c}'] = gdf_nh[c] * gdf_nh['frequency_period']

In [ ]:
for c in poi_cols:
    gdf_nh[f'co2_per_{c}'] = (
        gdf_nh[f'co2_{c}'] / gdf_nh[f'{c}_num']
    )

```python
totals = {
    "Education": 1200,
    "Healthcare and Health": 1523,
    "Recreational, Outdoors": 3809,
    "Shopping, Errands": 6280,
    "Social, Cultural": 7206,
```

In [ ]:
gdf_nh

In [ ]:
gdf_nh = gdf_nh.dropna(subset=['Nimi', 'Posnro'])

In [ ]:
co2_per_cols = [f'co2_per_{c}' for c in poi_cols]

nh_co2 = (
    gdf_nh
    .groupby('Nimi')
    .apply(lambda x: pd.Series({
        f'co2_per_{c}': x.loc[x[f'{c}_num'] > 0, f'co2_per_{c}'].mean()
        for c in poi_cols
    }))
    .reset_index()
)

In [ ]:
nh_co2.sort_values("co2_per_Social, Cultural").head(30)

In [ ]:
co2_per_cols = [f'co2_per_{c}' for c in poi_cols]

nh_co2_2 = (
    gdf_nh
    .groupby('Nimi')
    .apply(lambda x: pd.Series({
        f'co2_{c}': x.loc[x[f'{c}_num'] > 0, f'co2_{c}'].mean()
        for c in poi_cols
    }))
    .reset_index()
)

In [ ]:
nh_co2_2.sort_values("co2_Social, Cultural").head(30)

In [ ]:
nh_co2.sort_values("co2_per_Healthcare and Health").head(50)

In [ ]:
co2_cols = [f'co2_per_{c}' for c in poi_cols]

poi_cost = poi_cost.dropna(subset=co2_cols)

In [ ]:
co2_cols

In [ ]:
co2_per_cols = [f'co2_per_{c}' for c in poi_cols]

nimi_mean = (
    poi_cost
    .groupby('Nimi')[co2_per_cols]
    .mean()
    .reset_index()
)

In [ ]:
nimi_mean